## PART 1 — Introduction to Word Embeddings

**Task 1: Understanding Word Embeddings (Conceptual)**
Answer briefly:
1. What are word embeddings?
2. Why one-hot encoding and BoW fail to capture semantics?
3. How word embeddings solve these problems?

### task 1 answers

**1. what are word embeddings?**

basically dense vectors for words. each word becomes a list of numbers (like 100 floats) where similar words end up close in that space. not sparse 0/1 stuff like one-hot.

**2. why one-hot / bow fail at semantics?**

one-hot: every word is its own axis, all vectors are orthogonal so "king" and "queen" look as different as "king" and "pizza". no similarity.
bow/tfidf: just counts (or weighted counts). knows "film" showed up 3 times, doesnt know "film" ≈ "movie". also ignores order mostly.

**3. how embeddings fix that?**

words that show up in similar contexts get similar vectors. so distance/cosine actually means something. you can even do vector math kinda (king - man + woman ≈ queen). meaning gets baked into the numbers instead of treating vocab as disconnected ids.



## PART 2 — Word2Vec Overview & Techniques

**Task 2: Word2Vec Overview**
1. Explain what Word2Vec is.
2. Explain the idea of predicting words from context.
3. Define:
   - Vocabulary
   - Context window
   - Embedding dimension

### task 2 answers

**1. what is word2vec?**

a model (from google / mikolov) that learns word embeddings from raw text. gensim has it. you train on lots of sentences and each word gets a vector.

**2. predicting words from context**

idea is: words with similar neighbors should have similar meaning. so the model looks at a window around a word and tries to predict either the center word from neighbors, or neighbors from the center word. while doing that prediction task, the hidden weights become the embeddings.

**3. definitions**
- **vocabulary** — all unique words the model knows / was trained on
- **context window** — how many words left/right you look at (window=5 means up to 5 on each side)
- **embedding dimension** — length of each word vector (vector_size=100 → 100 numbers per word). bigger = more capacity but slower / needs more data


**Task 3: Types of Word2Vec Techniques**
Explain with examples:
1. CBOW (Continuous Bag of Words)
   - Predicts target word from context words
   - Faster and good for large datasets
2. Skip-Gram
   - Predicts context words from target word
   - Better for rare words
   - Write 3–4 lines explaining when to use each.

### task 3 answers

**CBOW** — continuous bag of words. you take the surrounding words and try to predict the middle one. eg context = [the, is, good] → predict "movie". usually faster, works ok when you have lots of data. averages context so its a bit blunt.

**Skip-Gram** — opposite direction. take the middle word and predict the context words around it. eg "movie" → predict the / is / good. slower usually, but better at rare words cos each rare word gets used as input more directly.

**when to use which (roughly)**
- lots of data, want speed → CBOW (sg=0)
- smaller data / care about rare words → Skip-Gram (sg=1)
- in practice i'd just try both and check similar words output, thats what the later tasks ask anyway



**Task 4: Neural Network Intuition Behind Word2Vec**
Explain (no coding required):
1. Input layer representation
2. Hidden layer (embedding layer)
3. Output layer
4. How weights become word embeddings

Extra (optional): Draw a simple diagram or explain in markdown.

### task 4 answers (nn intuition)

**1. input layer**
one-hot (or index) of the word(s). for CBOW its the context words, for skip-gram its the target word. super sparse.

**2. hidden layer (embedding layer)**
this is the important bit — a weight matrix basically (vocab_size × embedding_dim). multiplying one-hot by that matrix just picks the row for that word = the embedding vector. so the hidden layer *is* the embedding lookup.

**3. output layer**
tries to predict the word(s) — softmax over vocab (or negative sampling in practice so its not insanely slow). CBOW predicts target, skip-gram predicts context.

**4. how weights become embeddings**
you train with backprop on the prediction loss. the hidden weight matrix gets updated. after training, you throw away the output layer and just keep those hidden weights — each row is a word's embedding. thats the whole trick, the "side effect" of the prediction task is good vectors.

```
input (one-hot word)
        |
        v
  [ embedding weights ]  <-- keep this after training
        |
        v
   hidden vector (dense)
        |
        v
  output / softmax (predict other word)
```


## PART 3 — Training Word2Vec on Custom Data

**Task 5: Prepare Text for Word2Vec**

1. Tokenize cleaned text into sentences.
2. Tokenize sentences into words.
3. Store them as a list of lists.

Example:

```
[['this', 'movie', 'is', 'good'], ['i', 'liked', 'this', 'film']]
```

In [1]:
import pandas as pd 

In [2]:
df = pd.read_csv('tmdb_5000_movies.csv')

In [3]:
df = df[['overview']]
df.head()

,overview
0,"In the 22nd century, a paraplegic Marine is di..."
1,"Captain Barbossa, long believed to be dead, ha..."
2,A cryptic message from Bond’s past sends him o...
3,Following the death of District Attorney Harve...
4,"John Carter is a war-weary, former military ca..."


In [4]:
df.dropna(inplace=True)
df.head()

,overview
0,"In the 22nd century, a paraplegic Marine is di..."
1,"Captain Barbossa, long believed to be dead, ha..."
2,A cryptic message from Bond’s past sends him o...
3,Following the death of District Attorney Harve...
4,"John Carter is a war-weary, former military ca..."


In [5]:
import nltk

In [6]:
import re
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

In [7]:
stop_words = set(stopwords.words('english'))
wordnet_lemmatizer = WordNetLemmatizer()


In [8]:
def nlp_preprocess(text):
    # lowercasing
    text = text.lower()
    # noise removal
    # remove urls
    text = re.sub(r'https?://(www\.)?[-a-zA-Z0-9@:%._\+~#=]{2,256}\.[a-z]{2,4}\b([-a-zA-Z0-9@:%_\+.~#?&//=]*)', '', text)
    # remove email addresses
    text = re.sub(r'[\w\.-]+@[\w\.-]+', '', text)
    # remove html tags
    text = re.sub(r'<.*?>', '', text)
    # remove special characters & emojis
    text = re.sub(r'[^a-zA-Z0-9\s]', '', text)
    # remove extra spaces
    text = re.sub(r'\s+', ' ', text)
    # remove numbers
    text = re.sub(r'\d', '', text)
    # stopword removal
    text = ' '.join([word for word in text.split() if word not in stop_words])
    # tokenization
    tokens = word_tokenize(text)
    # lemmatization
    tokens = [wordnet_lemmatizer.lemmatize(word) for word in tokens]
    # join tokens back into text
    return tokens

In [9]:
df['cleaned_overview'] = df['overview'].apply(nlp_preprocess)
df.head()

,overview,cleaned_overview
0,"In the 22nd century, a paraplegic Marine is di...","[nd, century, paraplegic, marine, dispatched, ..."
1,"Captain Barbossa, long believed to be dead, ha...","[captain, barbossa, long, believed, dead, come..."
2,A cryptic message from Bond’s past sends him o...,"[cryptic, message, bond, past, sends, trail, u..."
3,Following the death of District Attorney Harve...,"[following, death, district, attorney, harvey,..."
4,"John Carter is a war-weary, former military ca...","[john, carter, warweary, former, military, cap..."


**Task 6: Train Word2Vec Model**
Using Gensim:

1. Train a Word2Vec model with:
   - vector_size = 100
   - window = 5
   - min_count = 1
   - sg = 0 (CBOW)
2. Print:
   - Vocabulary size
   - Embedding vector for a sample word


In [10]:
import gensim
from gensim.models import Word2Vec
import time

In [11]:
# train word2vec model
start_time_cbow = time.time()
word2vec_model = Word2Vec(df['cleaned_overview'], vector_size=100, window=5, min_count=1, sg=0)
end_time_cbow = time.time()

In [12]:
print(f"Vocabulary size: {len(word2vec_model.wv)}")

Vocabulary size: 20483


In [13]:
print(f"Embedding vector for 'movie': {word2vec_model.wv['movie']}")

Embedding vector for 'movie': [-0.30997008  0.79846185  0.39101803 -0.05962859  0.17640622 -1.0118281
  0.27611303  1.348542   -0.7148229  -0.55911016 -0.42536557 -1.0458785
 -0.18282507  0.11581225  0.1399789  -0.5315989  -0.24822585 -0.7215624
  0.02818505 -1.0442768   0.3552145   0.25611234  0.2636794  -0.28294358
 -0.14802818  0.03816144 -0.34522155 -0.34725955 -0.9526801  -0.3779364
  0.8404048   0.15628615  0.23965903 -0.51840675 -0.20842698  0.6914372
  0.31306607 -0.3715151  -0.6037025  -1.3387883   0.02219466 -0.75206333
 -0.33032277  0.04970472  0.27325267 -0.49481001 -0.2214821  -0.01323041
  0.42725947  0.4663736   0.36208043 -0.7296196  -0.527478   -0.31037343
 -0.5534997   0.39275053  0.42181554 -0.02622349 -0.45770246 -0.09559366
  0.14427903  0.6756938  -0.4180372  -0.28239867 -0.81537896  0.68829405
  0.24399716  0.59844697 -0.31615248  1.0083908  -0.57640433  0.3015968
  0.80963725 -0.17144336  0.894847    0.3961733   0.05052411 -0.03670714
 -0.9366229   0.31621054 -0

**Task 7: Skip-Gram Word2Vec Model**

1. Train another Word2Vec model using:
   - sg = 1 (Skip-Gram)

2. Compare:
   - Training time
   - Similar words output

In [14]:
start_time_sg = time.time()
word2vec_sg_model = Word2Vec(df['cleaned_overview'], vector_size=100, window=5, min_count=1, sg=1)
end_time_sg = time.time()



Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'


In [15]:
print(f"Training time (Skip-Gram): {end_time_sg - start_time_sg} seconds")
print(f"Training time (CBOW): {end_time_cbow - start_time_cbow} seconds")

Training time (Skip-Gram): 1.728987216949463 seconds
Training time (CBOW): 0.5139999389648438 seconds


In [16]:
# similar words
movie_similar_cbow = word2vec_model.wv.most_similar('movie')
movie_similar_sg = word2vec_sg_model.wv.most_similar('movie')

movie_similar_cbow_word = movie_similar_cbow[:5]
movie_similar_sg_word = movie_similar_sg[:5]

movie_similar_cbow_word_list = [word for word, _ in movie_similar_cbow_word]
movie_similar_sg_word_list = [word for word, _ in movie_similar_sg_word]

print(f"top 5 similar words to 'movie' (CBOW): {movie_similar_cbow_word_list}")
print(f"top 5 similar words to 'movie' (Skip-Gram): {movie_similar_sg_word_list}")



top 5 similar words to 'movie' (CBOW): ['relationship', 'family', 'american', 'becomes', 'meet']
top 5 similar words to 'movie' (Skip-Gram): ['drama', 'tale', 'comedy', 'follows', 'novel']


**Task 8: Word Similarity & Vector Operations**
Using the trained model:

1. Find most similar words to a given word.
2. Perform vector arithmetic:
    - king - man + woman = queen
    - Test similar relationships from your dataset.
    - Perform Other Operations mentioned in the videos

In [17]:
def find_most_similar_words(word, model, topn=5):
    similar_words = model.wv.most_similar(word, topn=topn)
    similar_words_list = [word for word, _ in similar_words]
    return similar_words_list

In [18]:
print(f"top 5 similar words to 'drama' (CBOW): {find_most_similar_words('drama', word2vec_model)}")
print(f"top 5 similar words to 'drama' (Skip-Gram): {find_most_similar_words('drama', word2vec_sg_model)}")


top 5 similar words to 'drama' (CBOW): ['film', 'american', 'movie', 'becomes', 'star']
top 5 similar words to 'drama' (Skip-Gram): ['tale', 'comedy', 'inspired', 'movie', 'epic']


In [19]:
king = word2vec_model.wv['king']
man = word2vec_model.wv['man']
woman = word2vec_model.wv['woman']

print(f"king - man + woman = {word2vec_model.wv.most_similar(king - man + woman, topn=1)}")



king - man + woman = [('woman', 0.9996209144592285)]


In [20]:
king_sg = word2vec_sg_model.wv['king']
man_sg = word2vec_sg_model.wv['man']
woman_sg = word2vec_sg_model.wv['woman']

print(f"king - man + woman = {word2vec_sg_model.wv.most_similar(king_sg - man_sg + woman_sg, topn=1)}")

king - man + woman = [('anna', 0.9788521528244019)]


In [21]:
# similar words
word2vec_model.wv.most_similar('drama')

[('film', 0.9997721314430237),
 ('american', 0.9997562766075134),
 ('movie', 0.9997439384460449),
 ('becomes', 0.9997396469116211),
 ('star', 0.9997294545173645),
 ('whose', 0.999721348285675),
 ('character', 0.9997149705886841),
 ('discovers', 0.9997105598449707),
 ('adventure', 0.9997096657752991),
 ('british', 0.9997019171714783)]

In [22]:
movie = word2vec_model.wv['movie']
drama = word2vec_model.wv['drama']

print(f"movie - drama = {word2vec_model.wv.most_similar(movie - drama, topn=1)}")

movie - drama = [('movie', 0.999615490436554)]


## PART 4 — Evaluation & Insights

**Task 9: Visualizing Word Embeddings (Optional)**

1. Reduce embedding dimensions using PCA or TSNE.
2. Plot word embeddings in 2D.
3. Observe clustering of similar words.


In [23]:
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go
from sklearn.decomposition import PCA
import numpy as np

In [24]:
# take top N most frequent words 
words = list(word2vec_model.wv.index_to_key)[:200]
vectors = np.array([word2vec_model.wv[w] for w in words])

# reduce 100-d -> 2-d
pca = PCA(n_components=2, random_state=42)
coords = pca.fit_transform(vectors)

In [25]:
# plotly scatter
fig = px.scatter(
    x=coords[:, 0],
    y=coords[:, 1],
    text=words,
    title="Word2Vec embeddings (PCA 2D)",
    labels={"x": "PC1", "y": "PC2"},
)
fig.update_traces(textposition="top center", marker=dict(size=8))
fig.update_layout(width=900, height=700)
fig.show()

**Task 10: Observations & Limitations**
Write short answers:

1. Difference between CBOW & Skip-Gram in practice
2. Advantages of Word2Vec over TF-IDF
3. Limitations of Word2Vec
4. Why context still matters in modern NLP (lead-in to transformers)

### task 10 answers

**1. cbow vs skip-gram in practice**

on my run skip-gram took longer than cbow (makes sense, its predicting more stuff). similar words for "movie" / "drama" were kinda related but not identical between the two — so they learn a bit different neighborhoods. cbow felt faster; skip-gram is what people say is better for rare words. for this movie-overview data both were usable, i wouldnt stress too hard which one unless timing matters.

**2. advantages of word2vec over tfidf**

tfidf is just weighted counts — "film" and "movie" are totally different features. word2vec puts them near each other in vector space if they show up in similar contexts. also fixed-size dense vectors instead of huge sparse bags. you can do weird vector math (king - man + woman) which tfidf just cant. so semantics > raw frequency counting.

**3. limitations of word2vec**

- one vector per word. "bank" (money) and "bank" (river) get the same embedding, no context
- needs decent amount of text or rare words are garbage (even with min_count=1 the vectors for rare words are meh)
- window size / hyperparams matter and theres no deep "understanding"
- doesnt handle unseen words well unless you do something extra
- still way behind contextual models (bert etc)

**4. why context still matters (→ transformers)**

meaning depends on the sentence. same word, different neighbors, different meaning. word2vec averages that into one static vector. transformers (bert/gpt) give a different vector for the word depending on the whole sentence, so they actually use context at prediction time. word2vec was a big step from bow/tfidf, but static embeddings are the limitation that contextual models fix.